# 01 — RecA–TB ChEMBL EC50 Data Collection and Curation

> Clean senior-review version. This notebook contains only the final, logically connected pipeline. 
> Exploratory/probing/failed scripts are intentionally excluded from the main workflow.

## Objective
Download/prepare RecA-related ChEMBL bioactivity data, standardize EC50 records, assign binary labels, and export the curated dataset for fingerprint generation.

In [ ]:
# Install only when running in a fresh Colab environment
# !pip install -q pandas numpy scikit-learn chembl_webresource_client

## 1. Imports and project paths

In [ ]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd()
RAW_DIR = PROJECT_DIR / "outputs" / "recA_chembl" / "raw"
CURATED_DIR = PROJECT_DIR / "outputs" / "recA_chembl" / "curated"

RAW_DIR.mkdir(parents=True, exist_ok=True)
CURATED_DIR.mkdir(parents=True, exist_ok=True)

TARGET_NAME = "RecA"
STANDARD_TYPE = "EC50"
ACTIVE_THRESHOLD_NM = 10000  # EC50 <= 10 µM = active-like

## 2. Load or download ChEMBL records

In [ ]:
# If you already have a ChEMBL export, place it here:
RAW_FILE = RAW_DIR / "recA_chembl_raw_ec50.csv"

if RAW_FILE.exists():
    raw_df = pd.read_csv(RAW_FILE)
else:
    from chembl_webresource_client.new_client import new_client

    target = new_client.target
    activity = new_client.activity

    targets = target.search(TARGET_NAME)
    target_ids = [t["target_chembl_id"] for t in targets if TARGET_NAME.lower() in str(t).lower()]

    records = []
    for target_id in target_ids:
        acts = activity.filter(target_chembl_id=target_id, standard_type=STANDARD_TYPE)
        records.extend(list(acts))

    raw_df = pd.DataFrame(records)
    raw_df.to_csv(RAW_FILE, index=False)

print(raw_df.shape)
raw_df.head()

## 3. Curate EC50 data and assign binary labels

In [ ]:
required = ["canonical_smiles", "standard_value", "standard_units", "molecule_chembl_id"]
missing = [c for c in required if c not in raw_df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = raw_df.copy()
df["standard_value"] = pd.to_numeric(df["standard_value"], errors="coerce")
df = df.dropna(subset=["canonical_smiles", "standard_value"])

# Keep nM records when unit information exists
if "standard_units" in df.columns:
    df = df[df["standard_units"].astype(str).str.lower().isin(["nm", "nanomolar"]) | df["standard_units"].isna()]

df = df[df["standard_value"] > 0].copy()
df["pEC50"] = 9 - np.log10(df["standard_value"])
df["bioactivity_class"] = np.where(df["standard_value"] <= ACTIVE_THRESHOLD_NM, 1, 0)

# Remove duplicate molecules using the median EC50/pEC50 value
curated = (
    df.groupby(["molecule_chembl_id", "canonical_smiles"], as_index=False)
      .agg(
          EC50_nM=("standard_value", "median"),
          pEC50=("pEC50", "median"),
          bioactivity_class=("bioactivity_class", "max"),
      )
)

curated = curated.rename(columns={"molecule_chembl_id": "Name", "canonical_smiles": "SMILES"})
curated["Name"] = curated["Name"].astype(str)

print(curated.shape)
print(curated["bioactivity_class"].value_counts())
curated.head()

## 4. Export final curated dataset

In [ ]:
CURATED_FILE = CURATED_DIR / "recA_curated_ec50_binary.csv"
curated.to_csv(CURATED_FILE, index=False)
print(f"Saved: {CURATED_FILE}")

## Final output
`outputs/recA_chembl/curated/recA_curated_ec50_binary.csv`